# Exploración de las viviendas de Barcelona

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Datos de entrada y problemas** de `SPECS.md`, y las ideas para tratarlo, a **Estrategias y experimentos**.

Aquí se mira y se piensa: nada de lo que hagáis en este notebook entra en el modelo. Las transformaciones que decidáis se escriben en `SPECS.md`, y el agente las implementa en `src/`, dentro del `Pipeline` (o en `limpiar_objetivo()`, si son sobre el precio), para que se apliquen igual en la validación cruzada, en la validación y en el test. Este notebook es el único sitio donde escribís código vosotros.

El significado de cada columna está en `DATOS.md`. Recordad: los barrios son reales, pero las viviendas y los precios son simulados.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # para importar src/ desde notebooks/

from src.datos import cargar_entrenamiento, cargar_test

entrenamiento = cargar_entrenamiento()
test = cargar_test()
print(f"Entrenamiento: {entrenamiento.shape[0]:,d} filas · Test: {test.shape[0]:,d} filas")

Entrenamiento: 6,237 filas · Test: 4,002 filas


## 1. Perfil de cada columna

In [2]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, ceros, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "ceros": (df == 0).sum(),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(entrenamiento)

,tipo,nulos,% nulos,ceros,distintos,ejemplo
fecha_venta,str,0,0.0,0,1355,17/02/2024
municipi,str,0,0.0,0,12,Barcelona
districte,str,1503,24.1,0,10,Sant Andreu
barri,str,1503,24.1,0,72,el Bon Pastor
direccion,str,183,2.9,0,4992,"Passeig de l'Havana, 16"
codigo_postal,float64,218,3.5,0,75,8030.0
latitud,float64,108,1.7,0,3225,41.43817
longitud,float64,108,1.7,0,3253,2.19852
distancia_centro_km,float64,0,0.0,0,161,6.2
ref_catastral,str,0,0.0,0,6084,9130140CM9461Z1396MC


In [15]:
entrenamiento["superficie_util"].agg(["min", "max"])

min     20.0
max    348.0
Name: superficie_util, dtype: float64

In [16]:
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

datos_superficie = entrenamiento[["superficie_util", "precio"]].dropna()
datos_superficie = datos_superficie[
    np.isfinite(datos_superficie["superficie_util"])
    & np.isfinite(datos_superficie["precio"])
]

X_superficie = datos_superficie[["superficie_util"]]
y_precio = datos_superficie["precio"]

modelos_superficie = {
    "Regresión lineal": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000)),
    "ElasticNet": make_pipeline(
        StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
    ),
}

tabla_metricas = []
x_linea_superficie = np.linspace(
    X_superficie["superficie_util"].min(),
    X_superficie["superficie_util"].max(),
    100,
).reshape(-1, 1)

figura_superficie = px.scatter(
    datos_superficie,
    x="superficie_util",
    y="precio",
    opacity=0.35,
    template="simple_white",
    title="Modelos lineales: superficie útil y precio",
    labels={"superficie_util": "Superficie útil (m²)", "precio": "Precio (€)"},
)

for nombre, modelo in modelos_superficie.items():
    modelo.fit(X_superficie, y_precio)
    predicciones = modelo.predict(X_superficie)

    tabla_metricas.append({
        "modelo": nombre,
        "MAE (€)": mean_absolute_error(y_precio, predicciones),
        "R²": r2_score(y_precio, predicciones),
    })

    figura_superficie.add_scatter(
        x=x_linea_superficie.ravel(),
        y=modelo.predict(x_linea_superficie),
        mode="lines",
        name=nombre,
    )

display(pd.DataFrame(tabla_metricas).set_index("modelo").round(3))
figura_superficie.show()

/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,MAE (€),R²
modelo,,
Regresión lineal,121363.294,0.544
Ridge,121360.510,0.544
Lasso,121363.237,0.544
ElasticNet,129538.771,0.484


In [17]:
columna = "superficie_util"

resumen_superficie = pd.DataFrame({
    "entrenamiento": entrenamiento[columna].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    ),
    "test": test[columna].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]
    ),
}).round(1)

display(resumen_superficie)
print(
    f"Nulos — entrenamiento: {entrenamiento[columna].isna().mean():.1%} · "
    f"test: {test[columna].isna().mean():.1%}"
)

datos_comparacion = pd.concat([
    entrenamiento[[columna]].assign(conjunto="Entrenamiento"),
    test[[columna]].assign(conjunto="Test"),
])

px.histogram(
    datos_comparacion.dropna(subset=[columna]),
    x=columna,
    color="conjunto",
    barmode="overlay",
    opacity=0.6,
    histnorm="probability density",
    template="simple_white",
    title="Superficie útil: entrenamiento y test",
    labels={columna: "Superficie útil (m²)"},
).show()

,entrenamiento,test
count,4191.0,2697.0
mean,76.9,76.8
std,34.0,34.9
min,20.0,22.0
10%,42.0,42.0
25%,54.0,55.0
50%,70.0,70.0
75%,92.0,91.0
90%,118.0,119.0
max,348.0,414.0


Nulos — entrenamiento: 32.8% · test: 32.6%


In [18]:
pd.DataFrame({
    "entrenamiento": entrenamiento["habitaciones"].agg(["min", "max"]),
    "test": test["habitaciones"].agg(["min", "max"]),
})

,entrenamiento,test
min,0,0
max,8,8


In [19]:
comparacion_habitaciones = pd.concat(
    [
        entrenamiento["habitaciones"].rename("Entrenamiento"),
        test["habitaciones"].rename("Test"),
    ],
    axis=1,
)

display(comparacion_habitaciones.describe().round(2))

frecuencias_habitaciones = pd.concat(
    [
        entrenamiento["habitaciones"].value_counts(normalize=True).rename("Entrenamiento"),
        test["habitaciones"].value_counts(normalize=True).rename("Test"),
    ],
    axis=1,
).fillna(0).sort_index().mul(100).round(1)

display(frecuencias_habitaciones.rename_axis("Habitaciones").rename_axis(columns="Conjunto"))

px.histogram(
    pd.concat(
        [
            entrenamiento[["habitaciones"]].assign(conjunto="Entrenamiento"),
            test[["habitaciones"]].assign(conjunto="Test"),
        ]
    ).dropna(subset=["habitaciones"]),
    x="habitaciones",
    color="conjunto",
    barmode="overlay",
    histnorm="probability",
    opacity=0.6,
    template="simple_white",
    title="Número de habitaciones: entrenamiento y test",
    labels={"habitaciones": "Habitaciones", "probability": "Proporción"},
).show()

,Entrenamiento,Test
count,6237.00,4002.00
mean,2.75,2.76
std,1.40,1.43
min,0.00,0.00
25%,2.00,2.00
50%,3.00,3.00
75%,3.00,4.00
max,8.00,8.00


Conjunto,Entrenamiento,Test
Habitaciones,,
0,4.6,5.4
1,10.8,10.8
2,29.5,27.4
3,30.3,31.1
4,15.3,15.3
5,5.5,5.7
6,2.4,2.8
7,1.6,1.1
8,0.1,0.3


In [20]:
datos_habitaciones = entrenamiento[["habitaciones", "precio"]].dropna()
datos_habitaciones = datos_habitaciones[
    np.isfinite(datos_habitaciones["habitaciones"])
    & np.isfinite(datos_habitaciones["precio"])
]

X_habitaciones = datos_habitaciones[["habitaciones"]]
y_habitaciones = datos_habitaciones["precio"]

modelos_habitaciones = {
    "Regresión lineal": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000)),
    "ElasticNet": make_pipeline(
        StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
    ),
}

tabla_habitaciones = []
x_linea_habitaciones = np.linspace(
    X_habitaciones["habitaciones"].min(),
    X_habitaciones["habitaciones"].max(),
    100,
).reshape(-1, 1)

figura_habitaciones = px.scatter(
    datos_habitaciones,
    x="habitaciones",
    y="precio",
    opacity=0.35,
    template="simple_white",
    title="Modelos lineales: habitaciones y precio",
    labels={"habitaciones": "Habitaciones", "precio": "Precio (€)"},
)

for nombre, modelo in modelos_habitaciones.items():
    modelo.fit(X_habitaciones, y_habitaciones)
    predicciones_habitaciones = modelo.predict(X_habitaciones)

    tabla_habitaciones.append({
        "modelo": nombre,
        "MAE (€)": mean_absolute_error(y_habitaciones, predicciones_habitaciones),
        "R²": r2_score(y_habitaciones, predicciones_habitaciones),
    })

    figura_habitaciones.add_scatter(
        x=x_linea_habitaciones.ravel(),
        y=modelo.predict(x_linea_habitaciones),
        mode="lines",
        name=nombre,
    )

display(
    pd.DataFrame(tabla_habitaciones)
    .set_index("modelo")
    .sort_values("MAE (€)")
    .round(3)
)
figura_habitaciones.show()

/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/tesssss/Desktop/Postgrau/Proyectos/MP2/mp2-equipo1/.venv/lib/python3.15/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,MAE (€),R²
modelo,,
Ridge,147624.854,0.346
Lasso,147627.746,0.346
Regresión lineal,147627.858,0.346
ElasticNet,148522.353,0.308


In [21]:
columnas_modelo = [
    "superficie_construida",
    "habitaciones",
    "banos",
    "distancia_centro_km",
    "poblacion_zona",
]

datos_modelo = entrenamiento[columnas_modelo + ["precio"]].dropna()
datos_modelo = datos_modelo[
    np.isfinite(datos_modelo[columnas_modelo + ["precio"]]).all(axis=1)
]

X_modelo = datos_modelo[columnas_modelo]
y_modelo = datos_modelo["precio"]

regresion_multiple = LinearRegression()
regresion_multiple.fit(X_modelo, y_modelo)
predicciones_modelo = regresion_multiple.predict(X_modelo)

print(f"Filas utilizadas: {len(datos_modelo):,}")
print(f"MAE en entrenamiento: {mean_absolute_error(y_modelo, predicciones_modelo):,.0f} €")
print(f"R² en entrenamiento: {r2_score(y_modelo, predicciones_modelo):.3f}")

coeficientes = pd.Series(
    regresion_multiple.coef_,
    index=columnas_modelo,
    name="coeficiente",
)
display(coeficientes.to_frame().round(2))
print(f"Intercepto: {regresion_multiple.intercept_:,.2f} €")

Filas utilizadas: 6,237
MAE en entrenamiento: 126,535 €
R² en entrenamiento: 0.451


,coeficiente
superficie_construida,1036.77
habitaciones,82317.82
banos,55075.46
distancia_centro_km,-12894.69
poblacion_zona,-0.57


Intercepto: 42,868.29 €


In [22]:
datos_particiones = entrenamiento[
    columnas_modelo + ["precio", "particion"]
].copy()

filas_finitas = np.isfinite(
    datos_particiones[columnas_modelo + ["precio"]]
).all(axis=1)
datos_particiones = datos_particiones.loc[filas_finitas]
datos_particiones = datos_particiones[
    datos_particiones["particion"].isin([1, 2, 3, 4, 5])
]

resultados_particiones = []

for particion in [1, 2, 3, 4, 5]:
    entrenamiento_fold = datos_particiones[
        datos_particiones["particion"] != particion
    ]
    validacion_fold = datos_particiones[
        datos_particiones["particion"] == particion
    ]

    modelo_fold = LinearRegression()
    modelo_fold.fit(
        entrenamiento_fold[columnas_modelo],
        entrenamiento_fold["precio"],
    )

    predicciones_fold = modelo_fold.predict(validacion_fold[columnas_modelo])
    precio_medio_fold = entrenamiento_fold["precio"].mean()
    predicciones_baseline = np.full(
        len(validacion_fold), precio_medio_fold
    )

    mae_modelo = mean_absolute_error(
        validacion_fold["precio"], predicciones_fold
    )
    mae_baseline = mean_absolute_error(
        validacion_fold["precio"], predicciones_baseline
    )

    resultados_particiones.append({
        "partición": particion,
        "MAE modelo (€)": mae_modelo,
        "MAE baseline (€)": mae_baseline,
        "mejora (€)": mae_baseline - mae_modelo,
    })

tabla_particiones = pd.DataFrame(resultados_particiones).set_index("partición")
display(tabla_particiones.round(0))

print(
    f"MAE media ± sd: "
    f"{tabla_particiones['MAE modelo (€)'].mean():,.0f} ± "
    f"{tabla_particiones['MAE modelo (€)'].std():,.0f} €"
)
print(
    f"Mejora media frente al baseline: "
    f"{tabla_particiones['mejora (€)'].mean():,.0f} €"
)

ValueError: Found array with 0 sample(s) (shape=(0, 5)) while a minimum of 1 is required by LinearRegression.

**¿Pasa lo mismo en el test?** El modelo tendrá que tasar esas 4.002 viviendas, con sus nulos y sus rarezas.

In [3]:
pd.DataFrame({
    "% nulos entrenamiento": (entrenamiento.isna().mean() * 100).round(1),
    "% nulos test": (test.isna().mean() * 100).round(1),
}).dropna(how="all").query("`% nulos entrenamiento` > 0 or `% nulos test` > 0")

,% nulos entrenamiento,% nulos test
any_construccio,29.8,30.1
ascensor,15.0,14.7
barri,24.1,23.1
certificado_energetico,8.1,8.4
codigo_postal,3.5,3.0
direccion,2.9,2.8
districte,24.1,23.1
estado,6.8,7.3
latitud,1.7,1.6
longitud,1.7,1.6


In [23]:
regresion_multiple = LinearRegression()
regresion_multiple.fit(X_modelo, y_modelo)

predicciones_modelo = regresion_multiple.predict(X_modelo)

print(f"MAE en entrenamiento: {mean_absolute_error(y_modelo, predicciones_modelo):,.0f} €")
print(f"R² en entrenamiento: {r2_score(y_modelo, predicciones_modelo):.3f}")

display(pd.Series(
    regresion_multiple.coef_,
    index=columnas_modelo,
    name="coeficiente",
).to_frame().round(2))

print(f"Intercepto: {regresion_multiple.intercept_:,.2f} €")

MAE en entrenamiento: 126,535 €
R² en entrenamiento: 0.451


,coeficiente
superficie_construida,1036.77
habitaciones,82317.82
banos,55075.46
distancia_centro_km,-12894.69
poblacion_zona,-0.57


Intercepto: 42,868.29 €


## 2. La variable objetivo

In [4]:
px.histogram(entrenamiento, x="precio", nbins=100, template="simple_white",
             title="Precio de la operación", labels={"precio": "Precio (€)"})

- ¿Qué forma tiene la distribución? ¿Cuáles son los precios más bajos y los más altos? ¿Alguno es imposible?
- Repetid el histograma con el logaritmo del precio. ¿Qué cambia? ¿Qué implicaría para un modelo lineal?
- `estado_operacion` dice si la operación se cerró. ¿Qué significa el precio de un anuncio **retirado**? ¿Es comparable con el de una venta?

In [5]:
entrenamiento.groupby("estado_operacion").agg(
    filas=("precio", "size"),
    precio_mediano=("precio", "median"),
    euros_m2_mediano=("precio", lambda p: (p / entrenamiento.loc[p.index, "superficie_construida"].replace(0, np.nan)).median()),
)

,filas,precio_mediano,euros_m2_mediano
estado_operacion,,,
retirada,620,332550.0,4211.881188
vendida,5617,282000.0,3496.666667


## 3. Preguntas de calidad para empezar

- ¿Hay superficies imposibles (0, enormes)? ¿Es siempre la construida mayor que la útil?
- ¿Qué valores toma `planta`? ¿Cómo la convertiríais en un número? ¿Qué es "Entlo" o "Pral"?
- ¿Qué valores toman `ascensor`, `parking` y `tipo`? ¿Cuántas formas distintas hay de decir lo mismo? ¿Qué significa "Opcional" en `parking`?
- ¿Qué significa un 0 en `terraza_m2`? ¿Y un vacío?
- ¿Hay años de construcción imposibles?
- ¿En qué formatos está `fecha_venta`?
- ¿Cuándo falta el barrio? ¿Siempre fuera de Barcelona?
- ¿Hay `ref_catastral` repetidas? ¿Es la misma operación dos veces o la misma vivienda vendida dos veces?
- `dias_en_venta`: ¿lo sabe el banco en el momento de tasar?
- ¿Cuántas operaciones tiene cada barrio y cada municipio? ¿Hay zonas del test que no aparezcan en el entrenamiento?

**Ejemplo:** los diez valores más frecuentes de `planta`

In [6]:
entrenamiento["planta"].value_counts(dropna=False).head(10)

planta
NaN          302
3º           248
2º           243
1º           231
4º           227
5º           217
Bajo         211
Ático        158
6º           143
1ª planta    139
Name: count, dtype: int64

**Ejemplo:** zonas del test que no están en el entrenamiento

In [7]:
zona_ent = entrenamiento["barri"].fillna(entrenamiento["municipi"])
zona_test = test["barri"].fillna(test["municipi"])
nuevas = ~zona_test.isin(zona_ent)
print(f"{nuevas.sum()} viviendas del test están en {zona_test[nuevas].nunique()} zonas sin operaciones en el entrenamiento")

4 viviendas del test están en 1 zonas sin operaciones en el entrenamiento


In [ ]:
# Vuestras comprobaciones aquí

## 4. Relación con el precio

Antes de probar una transformación, mirad cómo se relaciona cada variable con el precio. Una relación curva, valores imposibles o unos pocos puntos muy alejados son pistas de qué estrategia probar.

In [8]:
px.scatter(entrenamiento, x="superficie_construida", y="precio", color="municipi", opacity=0.4,
           template="simple_white", title="Precio frente a superficie construida",
           labels={"superficie_construida": "Superficie construida (m²)", "precio": "Precio (€)"})

- ¿Qué le pasa a este gráfico con ejes logarítmicos (`log_x=True, log_y=True`)?
- Repetidlo coloreando por `tipo` o por `estado`.
- Cread una columna con los €/m² y comparadla entre barrios: ¿cuánto varía?

In [9]:
entrenamiento.select_dtypes("number").corr(method="spearman")["precio"].sort_values()

distancia_centro_km     -0.400045
codigo_postal           -0.341811
longitud                -0.225743
latitud                 -0.219132
dias_en_venta           -0.040259
any_construccio         -0.029850
poblacion_zona           0.018046
terraza_m2               0.165615
banos                    0.507856
habitaciones             0.593337
superficie_construida    0.702580
superficie_util          0.713250
precio                   1.000000
Name: precio, dtype: float64

- ¿Por qué usamos la correlación de Spearman y no la de Pearson? (Sesión 06.)
- Una correlación baja, ¿quiere decir que la variable no sirve para tasar?

## 5. La ubicación

In [13]:
con_coordenadas = entrenamiento.dropna(subset=["latitud", "longitud"])
px.scatter_map(con_coordenadas, lat="latitud", lon="longitud", color=np.log10(con_coordenadas["precio"].clip(lower=1e4)),
               opacity=0.6, zoom=10, height=600, map_style="carto-positron",
               title="Precio según la ubicación", labels={"color": "log₁₀ precio"})

- ¿Dónde están las viviendas caras? ¿Ves algún punto fuera de sitio?
- La ubicación está en varias columnas: `municipi`, `districte`, `barri`, las coordenadas y `distancia_centro_km`. ¿Cuál usaríais en un modelo lineal y por qué?

## 6. Ideas para características nuevas

Pensad como un tasador: ¿qué miraría que no esté directamente en una columna? Algunas pistas: la altura real de la vivienda, si hay que subir escaleras, la época del edificio, el momento de la venta.

Escribid cada idea en `SPECS.md` §4 con su hipótesis **antes** de probarla.

In [ ]:
# Vuestras ideas aquí

In [14]:
variables_numericas = [
    columna
    for columna in entrenamiento.select_dtypes(include=["int64", "float64"]).columns
    if columna != "precio"
]

for variable in variables_numericas:
    datos = entrenamiento[[variable, "precio"]].dropna()
    datos = datos[np.isfinite(datos[variable]) & np.isfinite(datos["precio"])]

    figura = px.scatter(
        datos,
        x=variable,
        y="precio",
        title=f"Precio según {variable}",
        labels={variable: variable, "precio": "Precio (€)"},
        template="simple_white",
    )

    if len(datos) >= 2 and datos[variable].nunique() >= 2:
        pendiente, intercepto = np.polyfit(datos[variable], datos["precio"], 1)
        x_linea = np.linspace(datos[variable].min(), datos[variable].max(), 100)
        y_estimado = pendiente * x_linea + intercepto

        figura.add_scatter(
            x=x_linea,
            y=y_estimado,
            mode="lines",
            name="Regresión lineal",
        )

        y_real = datos["precio"].to_numpy()
        y_predicho = pendiente * datos[variable].to_numpy() + intercepto
        r2 = 1 - ((y_real - y_predicho) ** 2).sum() / ((y_real - y_real.mean()) ** 2).sum()

        print(f"{variable}: pendiente={pendiente:.2f}, R²={r2:.3f}")
    else:
        print(f"{variable}: no hay suficientes valores distintos para ajustar la recta.")

    figura.show()

codigo_postal: pendiente=-154.70, R²=0.040


latitud: pendiente=1694.05, R²=0.000


longitud: pendiente=-2376.79, R²=0.000


distancia_centro_km: pendiente=-10828.52, R²=0.017


superficie_construida: pendiente=2465.58, R²=0.246


superficie_util: pendiente=5983.76, R²=0.544


... (2 outputs left out)

terraza_m2: pendiente=5798.13, R²=0.035


any_construccio: pendiente=-55.02, R²=0.000


poblacion_zona: pendiente=-0.63, R²=0.025


dias_en_venta: pendiente=55.04, R²=0.000
